In [1]:
#------------------------------------------------ Begin_Librairie ----------------------------------------
# US FCA - Farm Credit Administration, FCS Institution Directory
# v1 (2026-05-26) superseded.  v2 was a cookie-debugging scratchpad, not a scraper.
#
# IMPORTANT - NETWORK REQUIREMENT
# Every FCA origin host (apps./reports./ww3./ww4.fca.gov, all on 4.79.206.0/24) refuses
# connections from outside the United States - the TCP handshake is dropped, not reset.
# Only www.fca.gov answers abroad, and only because it sits behind Cloudflare's CDN.
# So this scraper MUST run on a US-connected machine (US VPN, or the Windows control box).
# There is no code-side workaround for a dropped handshake.

import os
import re
import socket
import datetime
from time import sleep

import urllib3
import requests
import pandas as pd
from bs4 import BeautifulSoup

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)


In [2]:
#------------------------------------------------ Begin_fileName ----------------------------------------
regulatorName = 'US FCA'

print('Running {} Web Scraping Tool v.3.0'.format(regulatorName))

now = datetime.datetime.now()
processdate = now.strftime('%Y-%m-%d')
filename = '{} SQL Ready {}.xlsx'.format(regulatorName, str(now).replace(":", ".")[:-7])

# ------ At first we will define the workspace path -----
try:
    scriptfolder = os.path.dirname(os.path.abspath(__file__))  ## production environment (.py)
except NameError:
    scriptfolder = os.getcwd()  ## notebook environment

os.chdir(scriptfolder)

tempfolder = os.path.join(scriptfolder, 'tempfolder')
if os.path.exists(tempfolder):
    for rem in os.listdir(tempfolder):
        os.remove(os.path.join(tempfolder, rem))
else:
    os.mkdir(tempfolder)


Running US FCA Web Scraping Tool v.3.0


In [3]:
#------------------------------------------------ Begin_session ----------------------------------------
BASE = 'https://apps.fca.gov/FCSPublicDirectory/'
SEARCH_URL = BASE + 'PubSearchInstitution.aspx'

HEADERS = {
    'User-Agent': ('Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 '
                   '(KHTML, like Gecko) Chrome/125.0.0.0 Safari/537.36'),
    'Accept-Language': 'en-US,en;q=0.9',
}

session = requests.Session()
session.headers.update(HEADERS)
session.verify = False  # corporate TLS proxy


def preflight():
    """Fail loudly and specifically when the FCA origin is unreachable.

    Without this the scraper would emit a 0-row workbook that looks like a clean run.
    """
    try:
        socket.create_connection(('apps.fca.gov', 443), timeout=15).close()
    except OSError as exc:
        raise Exception(
            '\n[BLOCKED] : - cannot open a TCP connection to apps.fca.gov:443 ({}).\n'
            '            The FCA origin network (4.79.206.0/24) is geo-restricted to the US and\n'
            '            silently drops foreign handshakes. www.fca.gov answering is not a\n'
            '            counter-example - it is served by Cloudflare, the origin is not.\n'
            '            Re-run this notebook on a US VPN or on the Windows control machine.\n'
            '            No output file has been written.'.format(type(exc).__name__))
    print('[INFO] : - preflight OK, apps.fca.gov is reachable')


def get_soup(url, timeout=60, retries=3):
    for attempt in range(retries):
        try:
            resp = session.get(url, timeout=timeout)
            resp.raise_for_status()
            return BeautifulSoup(resp.text, 'html.parser')
        except Exception as exc:
            if attempt == retries - 1:
                raise
            print('[WARN] : - retry {}/{} on {} ({})'.format(attempt + 1, retries, url,
                                                             type(exc).__name__))
            sleep(3)


In [4]:
#------------------------------------------------ Begin_Dictionnary ----------------------------------------
# ListLabel : 1 = bank, 2 = insurance, 3 = bank & insurance, 4 = everything else.
# The Farm Credit System is a network of lending banks and agricultural credit
# associations, so the whole list is 1.
ListLabel = {'US FCA 1': 1}

Typology = {'US FCA 1': 'List of "Farm Credit System Institutions"'}

sqldict = {'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [],
          'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [],
          'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [],
          'RegCtry': [], 'RegCode' : [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],
          'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [],
          'Phone - Mother company': []}


In [5]:
#------------------------------------------------ Begin_Fonction ----------------------------------------

def bourange_same_length_array(sqldict):
    """Pad every column of sqldict out to the length of ListProcessDate."""
    maxlen = len(sqldict['ListProcessDate'])
    for key, val in sqldict.items():
        if len(sqldict[key]) != maxlen:
            empty = []
            total_empty = maxlen - len(sqldict[key])
            for i in range(total_empty):
                empty.append('')
            sqldict[key] = sqldict[key] + empty
    return sqldict


def clean(text):
    if text is None:
        return ''
    return ' '.join(str(text).split()).strip()


# Grid columns are located by header text, not by index - same reason the detail page is read
# by span id. A column inserted upstream would otherwise shift every field silently.
GRID_COLS = {
    'uninum':   'uninum',
    'name':     'institution name',
    'district': 'district',
    'state':    'hq state',
    'ceo':      'ceo',
    'rssd':     'rssd',
}


def parse_grid(soup):
    """Rows of the institution grid -> [{'name', 'uninum', 'district', 'state', 'ceo',
    'rssd', 'href'}].

    The grid is a single page - there is no pager - so a pager guard below asserts that
    assumption instead of trusting it.
    """
    grid = soup.find(id='ctl00_cphMainContent_gvInstitutions')
    if grid is None:
        raise Exception('[ERROR] : - grid #ctl00_cphMainContent_gvInstitutions not found. '
                        'Either the page was blocked, or the directory markup changed.')

    pager = [a for a in grid.find_all('a', href=True) if 'Page$' in a['href']]
    if pager:
        raise Exception('[ERROR] : - the grid is now paginated ({} pager links). This scraper '
                        'assumes a single page and would silently truncate.'.format(len(pager)))

    header = [clean(c.get_text()).lower() for c in grid.find('tr').find_all(['th', 'td'])]
    idx = {}
    for key, label in GRID_COLS.items():
        if label not in header:
            raise Exception('[ERROR] : - grid column "{}" is gone. Header is now: {}'
                            .format(label, header))
        idx[key] = header.index(label)

    out = []
    for tr in grid.find_all('tr'):
        link = tr.find('a', href=lambda h: h and 'PubViewInst.aspx' in h)
        if link is None:
            continue  # header / sort row
        tds = tr.find_all('td')
        rec = {key: clean(tds[pos].get_text()) if pos < len(tds) else ''
               for key, pos in idx.items()}
        rec['name'] = clean(link.get_text())          # the cell holds the anchor
        rec['href'] = BASE + link['href'].lstrip('/')
        out.append(rec)

    if not out:
        raise Exception('[ERROR] : - grid found but produced 0 institutions.')
    return out


# The detail page carries every value in a <span> whose id ends in a stable field name.
# v1 read them by row position (maintable[0]..maintable[6]), which breaks the moment FCA
# inserts or reorders a row - and breaks silently, shifting every field by one.
FIELD_IDS = {
    'uninum':      'lblUninum',
    'short_name':  'lblShortName',
    'status':      'lblStatusAndDesc',
    'phone':       'lblPhone',
    'rssd':        'lblRSSD',
    'ceo':         'lblCEO',
    'chairman':    'lblChairman',
    'address':     'lblCharterAddress',
    'county':      'lblCharterCounty',
    'website':     'hlWebURL',
    'official':    'lblInstName',
    'charterdate': 'lblCharterDate',
    'charternum':  'lblCharterNumber',
}


def parse_detail(soup):
    """Read the institution profile by span-id suffix, never by row position."""
    rec = {}
    for key, suffix in FIELD_IDS.items():
        el = soup.find(id=lambda i, s=suffix: bool(i) and i.endswith(s))
        if el is None:
            rec[key] = ''
            continue
        if key == 'address':
            # '30 E. 7th Street, Suite 700<br/>St. Paul, MN 55101-1810<br/>'
            rec[key] = [clean(p) for p in el.get_text('\n').split('\n') if clean(p)]
        elif key == 'website':
            rec[key] = clean(el.get('href') or el.get_text())
        else:
            rec[key] = clean(el.get_text())
    return rec


US_TAIL = re.compile(r'^(.*?),\s*([A-Z]{2})\s+([0-9]{5}(?:-[0-9]{4})?)\s*$')


def split_address(lines):
    """['30 E. 7th Street, Suite 700', 'St. Paul, MN 55101-1810']
       -> ('30 E. 7th Street, Suite 700', 'St. Paul', 'MN', '55101-1810')"""
    if not lines:
        return '', '', '', ''
    if len(lines) == 1:
        return lines[0], '', '', ''

    street = ', '.join(lines[:-1])
    tail = lines[-1]

    m = US_TAIL.match(tail)
    if m:
        return street, m.group(1).strip(), m.group(2), m.group(3)

    # no recognisable "City, ST ZIP" - keep the whole line as the city rather than
    # inventing a split
    return street, tail, '', ''


def usdate(text):
    """FCA publishes charter dates as m/d/yyyy."""
    text = clean(text)
    if not text:
        return ''
    for fmt in ('%m/%d/%Y', '%Y-%m-%d'):
        try:
            return datetime.datetime.strptime(text, fmt).strftime('%Y-%m-%d')
        except ValueError:
            continue
    return text


In [6]:
#------------------------------------------------ Begin_Main ----------------------------------------
preflight()

reg = 'US FCA 1'
listcode = '1'

print('[INFO] : - loading the institution grid')
institutions = parse_grid(get_soup(SEARCH_URL))
print('[INFO] : - {} institutions in the directory'.format(len(institutions)))

statuses = {}

for num, inst in enumerate(institutions, start=1):

    if num % 10 == 0 or num == len(institutions):
        print('[INFO] : - detail {}/{}'.format(num, len(institutions)))

    detail = parse_detail(get_soup(inst['href']))

    street, city, state, zipcode = split_address(detail.get('address') or [])

    status = detail.get('status', '')
    statuses[status] = statuses.get(status, 0) + 1

    sqldict['Name'].append(inst['name'])
    sqldict['EntryType'].append(detail.get('short_name', ''))
    sqldict['Typology'].append(inst['district'])
    sqldict['License_Type'].append(status)

    sqldict['InternalID_1'].append(detail.get('uninum') or inst['uninum'])
    sqldict['InternalID_1_type'].append('FCA Institution Number')
    sqldict['InternalID_2'].append(detail.get('rssd') or inst['rssd'])
    sqldict['InternalID_2_type'].append('RSSD Number')
    sqldict['InternalID_3'].append(detail.get('charternum', ''))
    sqldict['InternalID_3_type'].append('Charter Number')

    sqldict['Address_1'].append(street)
    sqldict['Address_2'].append(state or inst['state'])
    sqldict['City'].append(city)
    sqldict['Zip'].append(zipcode)
    sqldict['Cntry'].append('US')          # v1 wrote the US *state* into Cntry - fixed
    sqldict['Phone'].append(detail.get('phone', ''))
    sqldict['Website'].append(detail.get('website', ''))

    sqldict['Name - Mother Company'].append(detail.get('official', ''))
    sqldict['RegulationDate'].append(usdate(detail.get('charterdate', '')))

    sqldict['RegulationType'].append('Regulated')
    sqldict['RegCtry'].append('US')
    sqldict['RegCode'].append('FCA')
    sqldict['ListCode'].append(listcode)
    sqldict['ListName'].append(Typology[reg])
    sqldict['ListLabel'].append(ListLabel[reg])
    sqldict['ListLanguage'].append('EN')
    sqldict['ListProcessDate'].append(processdate)

    sqldict = bourange_same_length_array(sqldict)

# The directory is a positive register, so every row is emitted as 'Regulated'. If FCA ever
# starts publishing non-active institutions here, that is a mapping decision for the ticket
# owner rather than something to guess at - so surface it instead of burying it.
non_active = {k: v for k, v in statuses.items() if not k.lower().startswith('active')}
if non_active:
    print('[WARN] : - non-Active statuses present, all still emitted as RegulationType='
          "'Regulated' : {}".format(non_active))


Exception: 
[BLOCKED] : - cannot open a TCP connection to apps.fca.gov:443 (TimeoutError).
            The FCA origin network (4.79.206.0/24) is geo-restricted to the US and
            silently drops foreign handshakes. www.fca.gov answering is not a
            counter-example - it is served by Cloudflare, the origin is not.
            Re-run this notebook on a US VPN or on the Windows control machine.
            No output file has been written.

In [ ]:
#------------------------------------------------ Begin_writer and save df to excel ----------------------------------------
os.chdir(scriptfolder)

df = pd.DataFrame(sqldict)

df = df[df['Name'] != '']
df = df.drop_duplicates(subset=['Name', 'ListCode'], keep='first')
df = df.reset_index(drop=True)

# Refuse to write an empty workbook. A 0-row file looks like a successful run to whoever
# picks it up next - that is precisely how ES BES v1.2 shipped nothing for months.
if len(df) == 0:
    raise Exception('[ERROR] : - 0 rows collected, no file written. The scrape did not '
                    'complete - check the cells above for the blocking error.')

df.to_excel(os.path.join(scriptfolder, filename), sheet_name='SQL Ready', index=False)

print('Saved {} rows to {}'.format(len(df), os.path.join(scriptfolder, filename)))
print(df.groupby('ListCode').size())
